In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('model/features_engineered.csv')
print(f"Загружено {len(df)} игр с {len(df.columns)} признаками\n")

Загружено 72629 игр с 96 признаками



In [3]:
# Определяем рекомендованные признаки на основе корреляционного анализа
recommended_features = [
    'elo_diff', 'home_elo_before', 'away_elo_before',
    
    'win_rate_diff_L20', 'win_rate_diff_L10', 'win_rate_diff_L5',
    'home_win_rate_L20', 'home_win_rate_L10', 'home_win_rate_L5',
    'away_win_rate_L20', 'away_win_rate_L10', 'away_win_rate_L5',
    
    'fg_pct_diff_L20', 'fg_pct_diff_L10', 'fg_pct_diff_L5',
    'fg3_pct_diff_L20', 'fg3_pct_diff_L10', 'fg3_pct_diff_L5',
    
    'h2h_home_win_rate', 'h2h_total_games',
    
    'season_progress',
]

# Проверяем какие признаки доступны
available_features = [f for f in recommended_features if f in df.columns]
missing_features = [f for f in recommended_features if f not in df.columns]

print(f"Доступно признаков: {len(available_features)}")
if missing_features:
    print(f"Отсутствуют признаки: {missing_features}")
print(f"\nИспользуемые признаки:\n{available_features}")

Доступно признаков: 21

Используемые признаки:
['elo_diff', 'home_elo_before', 'away_elo_before', 'win_rate_diff_L20', 'win_rate_diff_L10', 'win_rate_diff_L5', 'home_win_rate_L20', 'home_win_rate_L10', 'home_win_rate_L5', 'away_win_rate_L20', 'away_win_rate_L10', 'away_win_rate_L5', 'fg_pct_diff_L20', 'fg_pct_diff_L10', 'fg_pct_diff_L5', 'fg3_pct_diff_L20', 'fg3_pct_diff_L10', 'fg3_pct_diff_L5', 'h2h_home_win_rate', 'h2h_total_games', 'season_progress']


In [4]:
# Подготовка X и y
X = df[available_features].copy()
y = df['home_win'].copy()

# Удаляем строки с пропущенными значениями
mask = ~(X.isna().any(axis=1) | y.isna())
X = X[mask]
y = y[mask]

print(f"После удаления пропущенных значений: {len(X)} игр")
print(f"Распределение целевой переменной: {y.value_counts().to_dict()}")
print(f"Win rate домашних команд: {y.mean():.3f}\n")

После удаления пропущенных значений: 50843 игр
Распределение целевой переменной: {1: 29890, 0: 20953}
Win rate домашних команд: 0.588



In [5]:
# Разделяем данные хронологически (важно для временных рядов)
# 80% для обучения, 20% для тестирования
split_idx = int(len(X) * 0.8)
X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(f"Обучающая выборка: {len(X_train)} игр")
print(f"Тестовая выборка: {len(X_test)} игр")
print(f"Train home win rate: {y_train.mean():.3f}")
print(f"Test home win rate: {y_test.mean():.3f}\n")

# Масштабируем признаки
print("Масштабирование признаков...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Масштабирование завершено\n")

Обучающая выборка: 40674 игр
Тестовая выборка: 10169 игр
Train home win rate: 0.598
Test home win rate: 0.547

Масштабирование признаков...
Масштабирование завершено



In [9]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, f1_score
import numpy as np

# Задаем сетку гиперпараметров для поиска
param_dist = {
    'max_depth': [None] + list(np.arange(2, 20)),
    'min_samples_split': np.arange(2, 20),
    'min_samples_leaf': np.arange(1, 20),
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_features': ['auto', 'sqrt', 'log2', None]
}

dt = DecisionTreeClassifier(random_state=42)

rs = RandomizedSearchCV(
    dt, 
    param_distributions=param_dist,
    n_iter=100,
    scoring='f1',
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

rs.fit(X_train_scaled, y_train)
print("Лучшие параметры:", rs.best_params_)
print(f"Лучшая кросс-валидационная accuracy: {rs.best_score_:.4f}")

# Предсказания на тесте
y_pred = rs.predict(X_test_scaled)
print("\nОценка на тестовой выборке:")
print(f"F1 score: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC score: {roc_auc_score(y_test, y_pred):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Лучшие параметры: {'min_samples_split': 4, 'min_samples_leaf': 8, 'max_features': None, 'max_depth': 3, 'criterion': 'gini'}
Лучшая кросс-валидационная accuracy: 0.7746

Оценка на тестовой выборке:
F1 score: 0.6339
ROC AUC score: 0.6043
Accuracy: 0.6339


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Задаем сетку гиперпараметров для поиска
rf_param_dist = {
    'n_estimators': np.arange(50, 301, 50),
    'max_depth': [None] + list(np.arange(2, 21, 2)),
    'min_samples_split': np.arange(2, 11, 2),
    'min_samples_leaf': np.arange(1, 11, 2),
    'criterion': ['gini', 'entropy', 'log_loss'],
}

rf = RandomForestClassifier(random_state=42)

rf_rs = RandomizedSearchCV(
    rf,
    param_distributions=rf_param_dist,
    n_iter=1,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

rf_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (Random Forest):", rf_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {rf_rs.best_score_:.4f}")

# Предсказания на тестовой выборке
rf_y_pred = rf_rs.predict(X_test_scaled)
rf_y_proba = rf_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(rf_rs, "predict_proba") else None

print("\nОценка на тестовой выборке (Random Forest):")
print(f"F1 score: {f1_score(y_test, rf_y_pred):.4f}")
if rf_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, rf_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, rf_y_pred):.4f}")

Fitting 5 folds for each of 1 candidates, totalling 5 fits
Лучшие параметры (Random Forest): {'n_estimators': 150, 'min_samples_split': 8, 'min_samples_leaf': 7, 'max_depth': 10, 'criterion': 'gini'}
Лучшая кросс-валидационная f1: 0.7667

Оценка на тестовой выборке (Random Forest):
F1 score: 0.7225
ROC AUC score: 0.7140
Accuracy: 0.6618


[CV 3/5] END max_depth=6, min_samples_split=6, n_estimators=50;, score=0.760 total time=   1.1s
[CV 5/5] END max_depth=6, min_samples_split=6, n_estimators=50;, score=0.759 total time=   1.1s
[CV 4/5] END max_depth=6, min_samples_split=6, n_estimators=50;, score=0.769 total time=   1.1s
[CV 1/5] END max_depth=6, min_samples_split=6, n_estimators=50;, score=0.758 total time=   1.1s
[CV 2/5] END criterion=gini, max_depth=10, min_samples_leaf=7, min_samples_split=8, n_estimators=150;, score=0.772 total time=   5.1s
[CV 5/5] END criterion=gini, max_depth=10, min_samples_leaf=7, min_samples_split=8, n_estimators=150;, score=0.756 total time=   5.1s
[CV 3/5] END criterion=gini, max_depth=10, min_samples_leaf=7, min_samples_split=8, n_estimators=150;, score=0.758 total time=   5.1s
[CV 1/5] END criterion=gini, max_depth=10, min_samples_leaf=7, min_samples_split=8, n_estimators=150;, score=0.777 total time=   5.1s
[CV 4/5] END criterion=gini, max_depth=10, min_samples_leaf=7, min_samples_split

Градиентный бустинг не получится обучить за один раз -- слишком долго. Сначала подберем для двух гиперпараметров, потом для остальных.

In [18]:
from sklearn.ensemble import GradientBoostingClassifier

# Задаем сетку гиперпараметров для поиска
gb_param_dist = {
    'n_estimators': np.arange(50, 301, 50),
    'max_depth': np.arange(2, 11, 2),
    # 'learning_rate': np.linspace(0.01, 0.3, 5),
    # 'subsample': np.linspace(0.6, 1.0, 3),
    # 'min_samples_split': np.arange(2, 11, 2),
    # 'min_samples_leaf': np.arange(1, 11, 2),
}

gb = GradientBoostingClassifier(random_state=42, verbose=3)

gb_rs = RandomizedSearchCV(
    gb,
    param_distributions=gb_param_dist,
    n_iter=50,
    scoring='f1',
    cv=5,
    verbose=3,  # Вывод процесса обучения
    random_state=42,
    n_jobs=-1
)

gb_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (Gradient Boosting):", gb_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {gb_rs.best_score_:.4f}")

# Предсказания на тестовой выборке
gb_y_pred = gb_rs.predict(X_test_scaled)
gb_y_proba = gb_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(gb_rs, "predict_proba") else None

print("\nОценка на тестовой выборке (Gradient Boosting):")
print(f"F1 score: {f1_score(y_test, gb_y_pred):.4f}")
if gb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, gb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, gb_y_pred):.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
      Iter       Train Loss   Remaining Time 
         1           1.3075           13.07s
         2           1.2753           12.23s
         3           1.2486           12.03s
         4           1.2263           11.98s
         5           1.2079           11.68s
         6           1.1919           11.49s
         7           1.1784           11.54s
         8           1.1667           11.35s
         9           1.1571           11.16s
        10           1.1489           11.16s
        11           1.1416           11.02s
        12           1.1350           10.87s
        13           1.1293           10.70s
        14           1.1244           10.58s
        15           1.1202           10.42s
        16           1.1162           10.35s
        17           1.1132           10.29s
        18           1.1100           10.20s
        19           1.1078           10.02s
        20           1.1050          

In [19]:
gb_rs.best_params_

{'n_estimators': 50, 'max_depth': 2}

In [22]:
gb = GradientBoostingClassifier(random_state=42, verbose=3, n_estimators=50, max_depth=2)

gb_param_dist = {
    'learning_rate': np.linspace(0.01, 0.3, 5),
    'min_samples_split': np.arange(2, 11, 2),
    'min_samples_leaf': np.arange(1, 11, 2),
}

gb_rs = RandomizedSearchCV(
    gb,
    param_distributions=gb_param_dist,
    n_iter=50,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-3
)

gb_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (Gradient Boosting):", gb_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {gb_rs.best_score_:.4f}")

# Предсказания на тестовой выборке
gb_y_pred = gb_rs.predict(X_test_scaled)
gb_y_proba = gb_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(gb_rs, "predict_proba") else None

print("\nОценка на тестовой выборке (Gradient Boosting):")
print(f"F1 score: {f1_score(y_test, gb_y_pred):.4f}")
if gb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, gb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, gb_y_pred):.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
        27           0.8957            1.47m
        28           0.8892            1.47m
        29           0.8783            1.47m
        30           0.8735            1.47m
        31           0.8679            1.46m
        32           0.8612            1.45m
        33           0.8550            1.45m
        34           0.8514            1.44m
        35           0.8484            1.44m
        36           0.8456            1.43m
        37           0.8430            1.43m
        38           0.8352            1.42m
        39           0.8318            1.41m
        40           0.8285            1.41m
        41           0.8240            1.40m
        42           0.8208            1.40m
        43           0.8162            1.39m
        44           0.8123            1.39m
        45           0.8088            1.38m
        46           0.8037            1.38m
        47           0.7993           

In [25]:
!pip3 install catboost

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 7.5 MB/s  0:00:03 eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]


        22           1.1089            1.72s
        23           1.1084            1.66s
        24           1.1077            1.60s
        25           1.1067            1.53s
        26           1.1060            1.48s
        27           1.1055            1.41s
        28           1.1051            1.35s
        29           1.1038            1.29s
        30           1.1023            1.23s
        31           1.1020            1.17s
        32           1.1016            1.11s
        33           1.1009            1.05s
        34           1.1006            0.99s
        35           1.1001            0.92s
        36           1.0994            0.86s
        37           1.0990            0.80s
        38           1.0981            0.74s
        39           1.0977            0.67s
        40           1.0971            0.61s
        41           1.0966            0.55s
        42           1.0963            0.49s
        43           1.0955            0.43s
        44

In [26]:
from catboost import CatBoostClassifier

catboost = CatBoostClassifier(
    random_seed=42,
    verbose=2,
    thread_count=-1
)

# Гиперпараметры для RandomizedSearchCV
catboost_param_dist = {
    'iterations': [100, 200, 300, 400],
    'depth': [3, 4, 5, 6],
    'learning_rate': np.linspace(0.01, 0.3, 5),
    'l2_leaf_reg': [1, 3, 5, 7, 9, 11],
}

catboost_rs = RandomizedSearchCV(
    catboost,
    param_distributions=catboost_param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

catboost_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (CatBoost):", catboost_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {catboost_rs.best_score_:.4f}")

cb_y_pred = catboost_rs.predict(X_test_scaled)
cb_y_proba = catboost_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(catboost_rs, "predict_proba") else None

print("\nОценка на тестовой выборке (CatBoost):")
print(f"F1 score: {f1_score(y_test, cb_y_pred):.4f}")
if cb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, cb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, cb_y_pred):.4f}")


Fitting 5 folds for each of 30 candidates, totalling 150 fits
0:	learn: 0.6246016	total: 62.2ms	remaining: 12.4s
2:	learn: 0.5717313	total: 82.2ms	remaining: 5.4s
4:	learn: 0.5614541	total: 102ms	remaining: 4s
6:	learn: 0.5563910	total: 121ms	remaining: 3.33s
8:	learn: 0.5512310	total: 137ms	remaining: 2.9s
10:	learn: 0.5485609	total: 196ms	remaining: 3.37s
12:	learn: 0.5471553	total: 244ms	remaining: 3.51s
14:	learn: 0.5461295	total: 281ms	remaining: 3.46s
16:	learn: 0.5446404	total: 333ms	remaining: 3.59s
18:	learn: 0.5418827	total: 347ms	remaining: 3.31s
20:	learn: 0.5405382	total: 368ms	remaining: 3.13s
22:	learn: 0.5395401	total: 387ms	remaining: 2.98s
24:	learn: 0.5384147	total: 401ms	remaining: 2.81s
26:	learn: 0.5373691	total: 418ms	remaining: 2.68s
28:	learn: 0.5360533	total: 434ms	remaining: 2.56s
30:	learn: 0.5348661	total: 453ms	remaining: 2.47s
32:	learn: 0.5335363	total: 470ms	remaining: 2.38s
34:	learn: 0.5323739	total: 491ms	remaining: 2.32s
36:	learn: 0.5314172	total: 

In [28]:
!pip3 install lightgbm

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 2.5 MB/s  0:00:00m 5.7 MB/s eta 0:00:01


In [30]:
!brew install libomp

⠋ JSON API formula.jws.json                          Downloading  32.0MB/-------
⠋ JSON API cask.jws.json                             Downloading  15.4MB/-------⠋ JSON API formula.jws.json                          Downloading  32.0MB/-------
⠋ JSON API cask.jws.json                             Downloading  15.4MB/-------⠙ JSON API formula.jws.json                          Downloading  32.0MB/-------
⠙ JSON API cask.jws.json                             Downloading  15.4MB/-------⠙ JSON API formula.jws.json                          Downloading  32.0MB/-------
⠙ JSON API cask.jws.json                             Downloading  15.4MB/-------⠚ JSON API formula.jws.json                          Downloading  32.0MB/-------
⠚ JSON API cask.jws.json                             Downloading  15.4MB/-------⠚ JSON API formula.jws.json                          Downloading  32.0MB/-------
⠚ JSON API cask.jws.json                             Downloading  15.4MB/-------⠞ JSON API formula.jws.json       

In [ ]:
from lightgbm import LGBMClassifier

lgb_param_dist = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [2, 4, 6, 8, -1],
    'learning_rate': np.linspace(0.01, 0.3, 5),
    'num_leaves': [15, 31, 63, 127],
    'min_child_samples': [5, 10, 20]
}

lgb = LGBMClassifier(random_state=42, n_jobs=-1)

lgb_rs = RandomizedSearchCV(
    lgb,
    param_distributions=lgb_param_dist,
    n_iter=30,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

lgb_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (LightGBM):", lgb_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {lgb_rs.best_score_:.4f}")

lgb_y_pred = lgb_rs.predict(X_test_scaled)
lgb_y_proba = lgb_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(lgb_rs, "predict_proba") else None

print("\nОценка на тестовой выборке (LightGBM):")
print(f"F1 score: {f1_score(y_test, lgb_y_pred):.4f}")
if lgb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, lgb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, lgb_y_pred):.4f}")

Fitting 5 folds for each of 30 candidates, totalling 150 fits
[LightGBM] [Info] Number of positive: 19464, number of negative: 13075
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001376 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3286
[LightGBM] [Info] Number of data points in the train set: 32539, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.598174 -> initscore=0.397865
[LightGBM] [Info] Start training from score 0.397865
[CV 3/5] END learning_rate=0.3, max_depth=-1, min_child_samples=10, n_estimators=150, num_leaves=63;, score=0.727 total time=  24.9s
[LightGBM] [Info] Number of positive: 19464, number of negative: 13075
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011858 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is

[LightGBM] [Info] Number of positive: 19464, number of negative: 13075
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001268 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3258
[LightGBM] [Info] Number of data points in the train set: 32539, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.598174 -> initscore=0.397865
[LightGBM] [Info] Start training from score 0.397865
[CV 2/5] END learning_rate=0.22749999999999998, max_depth=8, min_child_samples=5, n_estimators=50, num_leaves=31;, score=0.758 total time=   3.4s
[LightGBM] [Info] Number of positive: 19464, number of negative: 13075
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.013250 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3163
[LightGBM] [Info] Number of data points in the train set: 32539, number of used features: 21
[L

In [32]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV

# Параметры для RandomizedSearchCV
knn_param_dist = {
    'n_neighbors': list(range(1, 31)),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]  # 1 — манхэттен, 2 — евклид
}

knn = KNeighborsClassifier(n_jobs=-1)

knn_rs = RandomizedSearchCV(
    knn,
    param_distributions=knn_param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

knn_rs.fit(X_train_scaled, y_train)
print("Лучшие параметры (KNN):", knn_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {knn_rs.best_score_:.4f}")

knn_y_pred = knn_rs.predict(X_test_scaled)
knn_y_proba = knn_rs.predict_proba(X_test_scaled)[:, 1] if hasattr(knn_rs, "predict_proba") else None

print("\nОценка на тестовой выборке (KNN):")
print(f"F1 score: {f1_score(y_test, knn_y_pred):.4f}")
if knn_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, knn_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, knn_y_pred):.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Лучшие параметры (KNN): {'weights': 'distance', 'p': 2, 'n_neighbors': 27}
Лучшая кросс-валидационная f1: 0.7553

Оценка на тестовой выборке (KNN):
F1 score: 0.7090
ROC AUC score: 0.6953
Accuracy: 0.6512


In [33]:
from xgboost import XGBClassifier

# Параметры для RandomizedSearchCV для XGBoost
xgb_param_dist = {
    'n_estimators': [30, 50, 75, 100, 150],
    'max_depth': [2, 3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb = XGBClassifier(
    use_label_encoder=False, 
    eval_metric='logloss',
    n_jobs=-1,
    random_state=42
)

xgb_rs = RandomizedSearchCV(
    xgb,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring='f1',
    cv=5,
    verbose=3,
    random_state=42,
    n_jobs=-1
)

xgb_rs.fit(X_train, y_train)
print("Лучшие параметры (XGBoost):", xgb_rs.best_params_)
print(f"Лучшая кросс-валидационная f1: {xgb_rs.best_score_:.4f}")

xgb_y_pred = xgb_rs.predict(X_test)
xgb_y_proba = xgb_rs.predict_proba(X_test)[:, 1] if hasattr(xgb_rs, "predict_proba") else None

print("\nОценка на тестовой выборке (XGBoost):")
print(f"F1 score: {f1_score(y_test, xgb_y_pred):.4f}")
if xgb_y_proba is not None:
    print(f"ROC AUC score: {roc_auc_score(y_test, xgb_y_proba):.4f}")
else:
    print("ROC AUC score: N/A (predict_proba not available)")
print(f"Accuracy: {accuracy_score(y_test, xgb_y_pred):.4f}")

Fitting 5 folds for each of 20 candidates, totalling 100 fits


/Users/vladimirovdma/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [23:18:08] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/vladimirovdma/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [23:18:08] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/vladimirovdma/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [23:18:08] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
/Users/vladimirovdma/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [23:18:08] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are no

Лучшие параметры (XGBoost): {'subsample': 0.6, 'n_estimators': 100, 'max_depth': 2, 'learning_rate': 0.01, 'colsample_bytree': 0.8}
Лучшая кросс-валидационная f1: 0.7762

Оценка на тестовой выборке (XGBoost):
F1 score: 0.7325
ROC AUC score: 0.7096
Accuracy: 0.6483


Вывод: по итогам обучения и поиска различных гиперпараметров с помощью RandomizedSearchCV получили лучший результат по f1 score при XGBoost: 0.7325, accuracy при этом была лучшей при Catboost: 0.662.